# Proyek Mata Kuliah AI: Deteksi Misinformasi & Berita Hoaks Berbahasa Indonesia
### Model: Hybrid IndoBERT + BiLSTM + Attention Mechanism
Notebook ini disiapkan untuk melatih dan mengevaluasi model klasifikasi berita hoaks vs valid menggunakan representasi semantik mendalam dari **IndoBERT** dipadukan dengan **Bidirectional LSTM** dan **Attention Layer**.

In [ ]:
# 1. Install & Import Dependencies
!pip install -q transformers torch scikit-learn pandas openpyxl matplotlib seaborn

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

In [ ]:
# 2. Muat Data yang Telah Dipreprocessing
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
train_df.head(3)

In [ ]:
# 3. Model Architecture: Hybrid IndoBERT + BiLSTM + Attention
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        self.weight = nn.Linear(hidden_dim, hidden_dim)
        self.context_vector = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, rnn_outputs, mask=None):
        u = torch.tanh(self.weight(rnn_outputs))
        scores = self.context_vector(u).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask.bool(), -1e9)
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attention_weights.unsqueeze(1), rnn_outputs).squeeze(1)
        return context, attention_weights

class HybridIndoBERTBiLSTMAttention(nn.Module):
    def __init__(self, bert_model_name="indobenchmark/indobert-base-p1", lstm_hidden=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        self.lstm = nn.LSTM(768, lstm_hidden, bidirectional=True, batch_first=True)
        self.attention = BahdanauAttention(lstm_hidden * 2)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(outputs.last_hidden_state)
        context, weights = self.attention(lstm_out, mask=attention_mask)
        logits = self.classifier(context)
        return logits, weights
print("Model class berhasil didefinisikan!")

In [ ]:
# 4. Pipeline Uji Coba Prediksi Teks Tunggal
def predict_claim(text, model, tokenizer, max_length=128):
    model.eval()
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        logits, weights = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred = np.argmax(probs)
        
    label_map = {0: "VALID / FAKTA", 1: "HOAKS / MISINFORMASI"}
    return {
        "prediksi": label_map[pred],
        "probabilitas_fakta": float(probs[0]),
        "probabilitas_hoaks": float(probs[1])
    }
print("Pipeline inferensi siap.")